# Notebook 08 — Convergence and Cross-Validation Analysis

This notebook analyzes the production results from the SPINK7-KLK5 campaign:

1. **Structural analysis** — RMSD, RMSF, $R_g$, SASA from the 100 ns production trajectory.
2. **Jarzynski free energy** — $\Delta G$ from 50 SMD replicates via the Jarzynski equality.
3. **WHAM PMF** — Potential of mean force from 50 umbrella windows.
4. **Cross-validation** — Consistency check: $|\Delta G_{\text{Jarzynski}} - \Delta G_{\text{WHAM}}| < 2 k_B T$.
5. **Convergence curves** — $\Delta G$ vs. replicate count.

## Cross-Validation Criterion

At $T = 310$ K:

$$2 k_B T = 2 \times 0.008314 \times 310 \approx 5.15 \text{ kJ/mol} \approx 1.23 \text{ kcal/mol}$$

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import BOLTZMANN_KJ, DATA_DIR, KCAL_TO_KJ

ANALYSIS_DIR = DATA_DIR / "analysis"
PMF_DIR = ANALYSIS_DIR / "pmf"
PRODUCTION_DIR = DATA_DIR / "production"
print(f"Analysis directory: {ANALYSIS_DIR}")

## 1. Structural Analysis

Run the structural analysis CLI to compute RMSD, RMSF, $R_g$, and SASA
from the production trajectory.

```bash
python scripts/run_analysis.py structural \
    --trajectory data/production/production/production.dcd \
    --topology data/production/equilibration/topology_reference.pdb \
    --output data/analysis/structural_metrics.npz
```

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

structural_path = ANALYSIS_DIR / "structural_metrics.npz"

if structural_path.exists():
    data = np.load(structural_path)
    rmsd_nm = data["rmsd_nm"]
    n_frames = rmsd_nm.size
    time_ns = np.arange(n_frames) * 0.01  # 10 ps save interval

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(time_ns, rmsd_nm * 10.0, linewidth=0.5)  # nm -> Angstrom
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Backbone RMSD (\u00c5)")
    ax.set_title("SPINK7-KLK5 Production RMSD")
    ax.axhline(3.0, color="red", linestyle="--", label="3.0 \u00c5 threshold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "figures" / "rmsd_production.png", dpi=150)
    plt.show()

    # Report last 50 ns average.
    last_half = rmsd_nm[n_frames // 2:]
    print(f"RMSD (last 50 ns): {np.mean(last_half) * 10:.2f} \u00b1 "
          f"{np.std(last_half) * 10:.2f} \u00c5")
else:
    print(f"Structural metrics not found at {structural_path}.")
    print("Run: python scripts/run_analysis.py structural ...")

## 2. Jarzynski Free Energy Estimate

The Jarzynski equality connects non-equilibrium work $W$ to the equilibrium
free energy difference:

$$\Delta G = -k_B T \ln \left\langle e^{-\beta W} \right\rangle_{\text{NEQ}}$$

```bash
python scripts/run_analysis.py jarzynski \
    --work-values data/production/smd/total_work_values.npy \
    --temperature-k 310.0 \
    --output data/analysis/jarzynski_summary.npz
```

In [ ]:
jarz_path = ANALYSIS_DIR / "jarzynski_summary.npz"

if jarz_path.exists():
    jarz = np.load(jarz_path)
    dg_kj = float(jarz["delta_g_kj_mol"])
    dg_kcal = float(jarz["delta_g_kcal_mol"])
    print(f"Jarzynski \u0394G = {dg_kj:.2f} kJ/mol ({dg_kcal:.2f} kcal/mol)")
    print(f"Mean work = {float(jarz['mean_work_kj_mol']):.2f} kJ/mol")
    print(f"W_diss / k_BT = {float(jarz['w_diss_over_kbt']):.2f}")

    # Convergence plot.
    if "subset_sizes" in jarz and "delta_g_vs_n" in jarz:
        sizes = jarz["subset_sizes"]
        dg_vs_n = jarz["delta_g_vs_n"]
        std_vs_n = jarz["std_vs_n"]

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.errorbar(sizes, dg_vs_n, yerr=std_vs_n, fmt="o-", capsize=3)
        ax.set_xlabel("Number of SMD replicates")
        ax.set_ylabel("\u0394G (kJ/mol)")
        ax.set_title("Jarzynski Convergence")
        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / "figures" / "jarzynski_convergence.png", dpi=150)
        plt.show()
else:
    print(f"Jarzynski results not found at {jarz_path}.")
    print("Run: python scripts/run_analysis.py jarzynski ...")

## 3. WHAM Potential of Mean Force

The WHAM solver extracts the unbiased PMF $\mathcal{W}(\xi)$ from biased
umbrella windows:

$$\Delta G_{\text{bind}} = \mathcal{W}(\xi_{\text{bound}}) - \mathcal{W}(\xi_{\text{dissociated}})$$

```bash
python scripts/run_analysis.py wham \
    --xi-files data/production/umbrella/umbrella_window_*.npy \
    --window-centers $(python -c "...") \
    --spring-constants 1000.0 \
    --temperature-k 310.0 \
    --bootstrap \
    --output data/analysis/pmf/wham_pmf.npz
```

In [ ]:
wham_path = PMF_DIR / "wham_pmf.npz"

if wham_path.exists():
    wham = np.load(wham_path)
    xi_bins = wham["xi_bins"]
    pmf_kj = wham["pmf_kj_mol"]
    pmf_kcal = wham["pmf_kcal_mol"]

    dg_wham = float(np.min(pmf_kj) - pmf_kj[-1])
    print(f"WHAM \u0394G_bind = {dg_wham:.2f} kJ/mol "
          f"({dg_wham / KCAL_TO_KJ:.2f} kcal/mol)")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(xi_bins, pmf_kcal, "b-", linewidth=1.5)
    ax.set_xlabel("\u03be (nm)")
    ax.set_ylabel("PMF (kcal/mol)")
    ax.set_title("SPINK7-KLK5 PMF (WHAM)")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "figures" / "wham_pmf_production.png", dpi=150)
    plt.show()
else:
    print(f"WHAM PMF not found at {wham_path}.")
    print("Run: python scripts/run_analysis.py wham ...")

## 4. Cross-Validation

Compare the Jarzynski and WHAM $\Delta G$ estimates.
Both must agree within $2 k_B T \approx 5.15$ kJ/mol at $T = 310$ K.

```bash
python scripts/cross_validate.py \
    --jarzynski-npz data/analysis/jarzynski_summary.npz \
    --wham-npz data/analysis/pmf/wham_pmf.npz \
    --temperature-k 310.0
```

In [ ]:
if jarz_path.exists() and wham_path.exists():
    from scripts.cross_validate import cross_validate

    result = cross_validate(jarz_path, wham_path, temperature_k=310.0)

    print(f"Jarzynski \u0394G: {result['delta_g_jarzynski_kj_mol']:.2f} kJ/mol")
    print(f"WHAM \u0394G:      {result['delta_g_wham_kj_mol']:.2f} kJ/mol")
    print(f"Discrepancy:  {result['discrepancy_kj_mol']:.2f} kJ/mol")
    print(f"Threshold:    {result['threshold_kj_mol']:.2f} kJ/mol")
    print(f"Result:       {'PASSED' if result['passed'] else 'FAILED'}")
else:
    print("Both Jarzynski and WHAM results must exist for cross-validation.")
    print("Run the analysis workflows first.")